# Convolutional Neural Network Architecture

This section details the Convolutional Neural Network (CNN) architecture designed for magnetic field prediction. CNNs are specifically suited for processing structured grid data by exploiting spatial relationships through local connectivity and parameter sharing {cite}`lecun1998gradient,krizhevsky2012imagenet,goodfellow2016deep`.

**Key Architectural Innovations for Electromagnetic Field Prediction**:
- **Encoder-decoder structure** {cite}`ronneberger2015unet` for dense spatial regression
- **Dilated convolutions** {cite}`yu2015multi` to capture long-range electromagnetic interactions
- **Skip connections** to preserve fine-grained flux density details
- **Batch normalization** {cite}`ioffe2015batch` for stable training on FEA-generated data

## CNN Fundamentals

Convolutional Neural Networks are similar to ordinary neural networks in that they are made up of learnable weights and biases {cite}`lecun1998gradient,krizhevsky2012imagenet`. However, CNNs make an explicit assumption that **input data follows a structured pattern** to store information (usually in the form of a uniform grid such as images, time-series events, or magnetic field distributions).

This assumption introduces certain properties in the architecture which help in:
1. **Reducing the number of parameters** in the network (critical advantage over fully-connected ANNs)
2. **Making the training process easier** and more efficient through parameter sharing
3. **Exploiting spatial relationships** in the data through local connectivity

:::{important}
**Why CNNs for Electromagnetic Field Prediction**

Magnetic field distributions are naturally represented as 2D spatial grids:
- **Structured data**: Each pixel represents flux density at a specific (x, y) location
- **Spatial correlations**: Maxwell's equations dictate that field at one point depends on nearby regions (∇×B = μJ, ∇·B = 0)
- **Translation equivariance**: Physics laws apply uniformly across the spatial domain
- **Hierarchical patterns**: Local flux gradients → regional saturation zones → global field topology

**Quantitative advantage**: For a 256×256 field distribution:
- Fully-connected network (Section 4a): ~65 million weights in first layer alone
- CNN with 3×3 filters and 64 channels: Only **576 weights** per layer
- **Parameter reduction**: >100,000× fewer parameters while maintaining accuracy
:::

## Main Layer Types in CNN Architecture

There are three main types of layers present in CNN architecture for electromagnetic field prediction:

### 1. Convolutional Layer (CONV)

The convolutional layer computes the connection of each neuron with a local region of the input volume {cite}`lecun1998gradient`. It consists of a set of learnable filters (also called kernels) that extract spatial features.

```{figure} ../_static/figures/FIG2A.pdf
---
name: fig-conv-kernel
width: 60%
---
Convolution kernel visualization. Blue represents the input map, cyan represents the output map. The filter slides across the input, computing dot products.
```

**Key Properties**:
- **Small spatial extent**: Each filter is typically 3×3 or 5×5 (much smaller than input)
- **Sliding window**: Filter slides across entire input with stride (typically 1)
- **Dot product computation**: At each position, compute element-wise multiplication and sum
- **Feature map output**: 2D activation map showing filter response at each spatial location
- **Multiple filters**: Stack of K filters learns K different features, producing K feature maps

**Mathematical Operation**:

For a 2D convolution at position (i,j):
$$
Y[i,j] = \sum_{m} \sum_{n} X[i+m, j+n] \cdot K[m,n] + b
$$

where:
- $X$ is the input (e.g., geometry image or previous layer's feature maps)
- $K$ is the kernel/filter (learnable weights)
- $Y$ is the output feature map
- $b$ is the bias term (one per filter)

**Vectorized form** for entire spatial map:
$$
Y = X \ast K + b
$$

where $\ast$ denotes the convolution operation.

:::{note}
**Electromagnetic Interpretation**

What do convolutional filters learn for magnetic field prediction?

**Early layers** (close to input):
- **Local flux gradients**: Detect sharp changes in B-field magnitude
- **Material boundaries**: Identify transitions between air, iron core, magnets
- **Geometric features**: Recognize edges of slots, poles, magnets

**Middle layers**:
- **Saturation patterns**: Detect regions where core material saturates (nonlinear B-H)
- **Flux paths**: Learn typical flux return paths through iron
- **Interaction zones**: Recognize areas where multiple sources interact

**Deep layers**:
- **Global topology**: Understand overall field distribution symmetry
- **Source localization**: Relate excitation sources to far-field responses
- **Design patterns**: Learn relationships between geometry parameters and field shapes

This hierarchical feature learning mirrors the multi-scale nature of electromagnetic phenomena—from local Ampere's law (∇×H = J) to global flux conservation (∮B·dA = 0).
:::

**Parameter Sharing**:
- Same filter applied at every spatial location
- A 3×3 filter has only 9 weights (plus 1 bias) regardless of input size
- For 64 filters: 64 × (9+1) = **640 parameters** vs. millions for fully-connected layer

### 2. Pooling Layer (POOL)

The pooling layer is responsible for downsampling the spatial dimensions of the input {cite}`lecun1998gradient,goodfellow2016deep`. Pooling layers are periodically inserted in-between convolutional layers to progressively reduce spatial resolution.

**Purpose**:
- **Reduce parameters**: Smaller spatial dimensions → fewer computations in subsequent layers
- **Increase receptive field**: Each neuron "sees" a larger region of the original input
- **Provide translation invariance**: Small shifts in input don't change output significantly
- **Control overfitting**: Reduces model capacity, preventing memorization of training geometries

**Most Popular Form**: Max pooling with $2 \times 2$ filters applied with stride of 2:
- Down-samples every depth slice by 2 along both width and height
- **Dimension reduction**: H×W → H/2×W/2 (75% spatial reduction)
- **Channel preservation**: Depth dimension remains unchanged
- **Operation**: $Y[i,j] = \max(X[2i:2i+2, 2j:2j+2])$ for each channel

**Hyperparameters**:
1. **Filter Size (F)**: Typically $2 \times 2$ or $3 \times 3$
2. **Stride (S)**: Step size for sliding the filter (typically 2 for max pooling)

**Output dimensions**:
$$
H_{\text{out}} = \lfloor (H_{\text{in}} - F) / S \rfloor + 1
$$
$$
W_{\text{out}} = \lfloor (W_{\text{in}} - F) / S \rfloor + 1
$$

**Example**: A $4 \times 4$ input with $2 \times 2$ max pooling (stride 2) produces a $2 \times 2$ output:

```
Input (4×4):                    Output (2×2) after max pooling:
[1  3  | 2  4]                  [3 | 4]
[5  6  | 1  2]                  [6 | 2]
-------+------                  ---+---
[7  2  | 8  3]                  [7 | 8]
[1  0  | 4  1]                  [1 | 4]

Takes max of each 2×2 block
```

:::{note}
**Trade-offs for Electromagnetic Field Prediction**

**Advantages**:
- **Computational efficiency**: Training and inference speed increases significantly
- **Receptive field growth**: Deeper layers can "see" larger portions of the motor geometry
- **Abstraction**: Forces network to learn high-level electromagnetic patterns

**Disadvantages**:
- **Spatial information loss**: Precise flux density locations become blurred
- **Boundary details**: Sharp transitions (air-iron interfaces) may be smeared
- **Resolution reduction**: Difficult to predict fine-grained field variations

**Solution in this work**: The encoder-decoder architecture with **skip connections** (Section below) recovers lost spatial information during upsampling, combining coarse semantic features from encoder with fine spatial details from early layers.
:::

### 3. Fully Connected Layer (FC)

Neurons in fully connected layers have dense connections with all activations in the previous layer, similar to regular neural networks.

**Differences from CONV Layers**:
- **Structure**: CONV layers exhibit local connectivity and parameter sharing
- **Function**: Both compute dot products between weights and activations, followed by non-linear activation

**Purpose**: 
- Balance the locality-context tradeoff
- Relate all features learned by CONV layers
- Develop patterns that capture global trends

**Note**: In modern encoder-decoder architectures for dense prediction (like field estimation), FC layers are often omitted in favor of fully convolutional architectures.

## Dilated Convolutions

A key innovation in the magnetic field prediction architecture is the use of **dilated convolutions** (also called atrous convolutions) {cite}`yu2015multi`. This technique is critical for capturing long-range electromagnetic interactions without increasing computational cost.

### Standard vs. Dilated Convolution

**Standard Convolution**: Dense kernel captures only immediate neighbors

```{figure} ../_static/figures/FIG2A.pdf
---
name: fig-standard-conv
width: 45%
---
Non-dilated convolutional filter with padding. All kernel elements are contiguous.
```

**Dilated Convolution**: Sparse kernel captures distant relationships

```{figure} ../_static/figures/FIG2B.pdf
---
name: fig-dilated-conv
width: 45%
---
Dilated filter with dilation rate of 1. Kernel elements are spaced apart, expanding the receptive field without increasing parameters.
```

### Mathematical Formulation

**Standard convolution**:
$$
Y[i,j] = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} X[i+m, j+n] \cdot K[m,n]
$$

**Dilated convolution** with dilation rate $r$:
$$
Y[i,j] = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} X[i+r \cdot m, j+r \cdot n] \cdot K[m,n]
$$

**Receptive field**: Effective filter size becomes $(k-1) \cdot r + 1$
- $r=1$ (standard): 3×3 filter has receptive field of 3×3 = 9 pixels
- $r=2$ (dilation=1): 3×3 filter has receptive field of 5×5 = 25 pixels
- $r=3$ (dilation=2): 3×3 filter has receptive field of 7×7 = 49 pixels

**Key advantage**: Exponential receptive field growth without parameter increase!

### Motivation for Magnetic Field Prediction

**Physical Insight from Electromagnetic Theory**:

The magnetic field at a point is determined by the **Biot-Savart law**:
$$
\mathbf{B}(\mathbf{r}) = \frac{\mu_0}{4\pi} \int \frac{\mathbf{J}(\mathbf{r'}) \times (\mathbf{r} - \mathbf{r'})}{|\mathbf{r} - \mathbf{r'}|^3} dV'
$$

This shows that **the field at location $\mathbf{r}$ depends on current density $\mathbf{J}$ at ALL locations $\mathbf{r'}$**—including spatially distant regions. The $1/|\mathbf{r} - \mathbf{r'}|^2$ falloff means distant sources still contribute significantly.

**Practical implications**:
- **Airgap flux** in a motor is influenced by rotor magnets, stator windings, AND iron saturation in the back iron (cm away)
- **Fringing fields** extend far from slot openings
- **Mutual coupling** between coils separated by multiple slots
- **Leakage flux paths** through air can span the entire motor diameter

**Why standard 3×3 convolutions are insufficient**:
- With stride=1 and no pooling, receptive field after $N$ layers is only $(2N+1) \times (2N+1)$
- For 10 layers: receptive field = 21×21 pixels
- For a 256×256 motor image: Only sees 8% of spatial dimension
- To see entire motor would need ~128 layers (infeasible to train)

**Why dilated convolutions solve this** {cite}`yu2015multi`:
- Alternating dilated layers exponentially expand receptive field
- After 10 layers with alternating dilation [1,2,1,2,...]: receptive field ~200×200 pixels
- Sees ~78% of spatial domain with 10× fewer layers
- **Same parameter count** as standard convolutions (still 3×3 filters)

:::{important}
**Empirical Results from This Work**

Ablation studies (Section 4d) demonstrate:

| Architecture | Test NMSE | Receptive Field |
|--------------|-----------|-----------------|
| Standard CNN (no dilation) | 1.2% | ~60×60 pixels |
| CNN with dilated layers | **0.4%** | ~200×200 pixels |
| Improvement | **3× better** | **11× larger** |

**Conclusion**: Dilated convolutions are **essential** for accurate electromagnetic field prediction. Standard CNNs cannot capture the long-range interactions dictated by Maxwell's equations without prohibitively deep architectures.
:::

### Implementation Details

In this work:
- **Dilation pattern**: Alternating [1, 2, 1, 2, ...] in encoder layers
- **Kernel size**: 3×3 (compact even with dilation)
- **Placement**: Applied in encoder layers 2, 4, 6 (after initial feature extraction)
- **Decoder**: Standard convolutions (dilation not needed for reconstruction)

## Encoder-Decoder Architecture

The deep learning model for magnetic field prediction uses an **encoder-decoder architecture** {cite}`ronneberger2015unet`, similar to the U-Net architecture designed for medical image segmentation but adapted for dense regression instead of classification.

```{figure} ../_static/figures/Network_arch.png
---
name: fig-network-arch
width: 95%
---
Encoder-Decoder based Convolutional Neural Network Architecture for magnetic field prediction. The encoder (left half) progressively downsamples to extract abstract features. The decoder (right half) upsamples to reconstruct the spatial field distribution. Skip connections (horizontal arrows) preserve fine-grained details.
```

### Encoder Section (Contracting Path)

The encoder extracts hierarchical features from the input geometry representation:

**Progressive abstraction** (left side of network):
1. **Layer 1-2** (256×256 → 128×128): Local geometric features
   - Slot edges, magnet boundaries, winding positions
   - Learns: "Where are the materials located?"
   
2. **Layer 3-4** (128×128 → 64×64): Regional electromagnetic patterns
   - Flux concentration zones, saturation regions
   - Learns: "Where will flux be high/low?"
   
3. **Layer 5-6** (64×64 → 32×32): Global field topology
   - Flux return paths, symmetry patterns
   - Learns: "What is the overall field structure?"
   
4. **Bottleneck** (32×32): Compressed latent representation
   - Most discriminative features for this geometry
   - Learns: "What are the key electromagnetic relationships?"

**Technical components per encoder block**:
- Two $3 \times 3$ convolutional layers (some with dilation)
- ReLU activation after each convolution {cite}`nair2010rectified`
- Batch normalization for stable training {cite}`ioffe2015batch`
- $2 \times 2$ max-pooling with stride 2 (75% spatial reduction)
- Dropout (rate=0.5) for regularization {cite}`srivastava2014dropout`

**Feature channel growth**: 32 → 64 → 128 → 256 as spatial dimensions shrink
- Compensates for spatial information loss with richer feature representations
- Total information capacity remains approximately constant

### Decoder Section (Expanding Path)

The decoder reconstructs the full-resolution field distribution from the compressed latent representation:

**Progressive reconstruction** (right side of network):
1. **Bottleneck → Layer 7-8** (32×32 → 64×64): Coarse field structure
   - Major flux paths, high-level topology
   - Reconstructs: "Overall field shape"
   
2. **Layer 9-10** (64×64 → 128×128): Regional field details
   - Saturation zone boundaries, flux gradients
   - Reconstructs: "Regional variations"
   
3. **Layer 11-12** (128×128 → 256×256): Fine-grained field distribution
   - Precise flux density at each spatial location
   - Reconstructs: "Pixel-level predictions"

**Technical components per decoder block**:
- $2 \times 2$ transposed convolution (upsampling) with stride 2
- Concatenation with corresponding encoder feature maps (skip connections)
- Two $3 \times 3$ convolutional layers
- ReLU activation and batch normalization
- Dropout for regularization

**Feature channel reduction**: 256 → 128 → 64 → 32 → 1 as spatial dimensions grow
- Progressively focuses on spatial precision over feature diversity
- Final layer: 1 channel = predicted flux density magnitude

### Skip Connections (Critical Innovation)

Skip connections directly connect encoder layers to corresponding decoder layers {cite}`ronneberger2015unet`:

**Why skip connections are essential**:

**Problem**: Spatial information is lost during encoder downsampling
- Max-pooling discards 75% of spatial positions at each step
- After 4 pooling operations: 256×256 → 16×16 (99.6% spatial reduction)
- Decoder must reconstruct field from highly compressed representation
- **Result without skip connections**: Blurry predictions, poor boundary localization

**Solution**: Concatenate encoder feature maps with decoder feature maps
```
Decoder Layer 9 input = [Upsampled Layer 8] ⊕ [Encoder Layer 3]
                        (low-res features)    (high-res features)
```

where ⊕ denotes channel-wise concatenation.

**Benefits**:
1. **Precise localization**: Encoder features preserve exact positions of geometric boundaries
2. **Sharp transitions**: Air-iron interfaces, magnet edges remain crisp
3. **Gradient flow**: Direct paths from output to early layers improve training
4. **Multi-scale fusion**: Combines semantic (what) and spatial (where) information

:::{tip}
**Electromagnetic Context**

Skip connections are particularly important for magnetic field prediction because:

- **Material boundaries**: Sharp flux density transitions at air-iron interfaces (B changes by 1000×) must be precisely localized
- **Slot openings**: Fringing fields have strong spatial gradients requiring accurate positioning
- **Airgap flux**: Small airgap (1-2mm) needs sub-pixel accuracy to predict torque correctly
- **Saturation zones**: Exact spatial extent of core saturation affects overall performance

**Quantitative impact** (ablation study, Section 4d):
- With skip connections: 0.4% NMSE, sharp boundaries
- Without skip connections: 2.1% NMSE, blurry boundaries
- **5× accuracy improvement** from skip connections alone!
:::

### Comparison to U-Net

This architecture is inspired by U-Net {cite}`ronneberger2015unet` but adapted for electromagnetic field prediction:

| Aspect | U-Net (Medical Imaging) | This Work (EM Fields) |
|--------|------------------------|----------------------|
| **Task** | Segmentation (classification) | Regression (continuous values) |
| **Output** | Class labels per pixel | Flux density per pixel |
| **Output activation** | Softmax | Linear (no activation) |
| **Loss function** | Cross-entropy | Mean Squared Error |
| **Innovation added** | - | **Dilated convolutions** for long-range EM interactions |
| **Receptive field** | Standard | **Exponentially expanded** via dilation |

**Key insight**: U-Net's architecture is well-suited for any dense prediction task requiring both semantic understanding (encoder) and precise localization (decoder + skip connections).

## Network Specifications

The complete deep learning model consists of:

| Component | Count |
|-----------|-------|
| **Total Layers** | 32 |
| **Trainable Convolutional Layers** | 16 |
| **Pooling/Up-sampling Layers** | 8 |
| **Dropout Layers** | 8 |
| **Total Parameters** | 2.4 Million |

### Layer Configuration

Each encoder/decoder block consists of:
1. Two sets of $3 \times 3$ convolution layers
2. ReLU activation after each convolution
3. Batch normalization layer {cite}`ioffe2015batch`
4. $2 \times 2$ max-pooling (encoder) or up-sampling (decoder) with stride 2
5. Dropout for regularization {cite}`srivastava2014dropout` (typically 0.5 dropout rate)

**Exception**: The last two convolutional layers do not follow this pattern due to a network design constraint to fit the field distribution dimensionality.

### Convolution Details

- **Kernel Size**: $3 \times 3$ (with some $5 \times 5$)
- **Stride**: 1 for all convolutional layers
- **Number of Filters (K)**: Varies by layer, typically K=32 or K=64
- **Dilation**: Alternate dilated layers in encoder with dilation rate of 1

## Model Capacity and Generalization

For the network to train successfully, it must have **sufficient capacity** to satisfactorily capture the complexity of the electromagnetic field prediction problem. This capability is achieved by choosing appropriate values for {cite}`goodfellow2016deep`:

1. **Number of Layers**: Depth of the network (32 total layers)
2. **Kernel Size**: Size of convolutional filters ($3 \times 3$, $5 \times 5$)
3. **Number of Kernels**: Filters per layer (K=32, K=64)
4. **Regularization Parameters**: Dropout rate, batch normalization

### Model Selection Process

Model selection is performed by {cite}`bergstra2012random`:
- Investigating a set of candidate models
- Choosing the model with best balance between fit and complexity
- Using random search and grid search for hyperparameter optimization

**Result**: A total of 35 networks with different configurations were trained during the model selection process before arriving at the final architecture described here.

:::{note}
**Understanding This Visualization**  
This demonstration shows how a convolutional filter processes input data:

1. **Input (5×5)**: Could represent a small patch of a magnetic field or geometry image
2. **Kernel (3×3)**: Learnable filter that detects specific patterns (e.g., edges, gradients)
3. **Feature Map (3×3)**: Output showing where the pattern is detected (high values = strong match)

The red box on the input shows the **receptive field**—the region that influences one output pixel. The filter slides across the entire input, computing the dot product at each position.

**For electromagnetic field prediction**: Early-layer filters learn to detect geometric features (slot edges, magnet boundaries), while deeper filters learn electromagnetic patterns (flux concentration zones, saturation regions).
:::

In [ ]:
# Visualization of CNN architecture components

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch, FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap

def visualize_convolution_detailed():
    """
    Comprehensive demonstration of convolution operation with step-by-step breakdown.
    """
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # Create sample input (simulating a magnetic field patch)
    np.random.seed(42)
    input_data = np.random.rand(5, 5)
    # Add some structure (simulate flux gradient)
    input_data[1:4, 1:4] += 0.5
    
    # Define an edge detection kernel (simulates learning flux gradients)
    kernel = np.array([[-1, 0, 1],
                       [-2, 0, 2],
                       [-1, 0, 1]]) / 4.0  # Sobel-like filter
    
    # Compute actual convolution (valid padding)
    output_size = 3
    output_data = np.zeros((output_size, output_size))
    
    for i in range(output_size):
        for j in range(output_size):
            patch = input_data[i:i+3, j:j+3]
            output_data[i, j] = np.sum(patch * kernel)
    
    # ===== Top Row: Input, Kernel, Output =====
    ax1 = fig.add_subplot(gs[0, 0])
    im1 = ax1.imshow(input_data, cmap='viridis', interpolation='nearest')
    ax1.set_title('Input Feature Map (5×5)\n(Could be flux density or geometry)', 
                  fontweight='bold', fontsize=11)
    
    # Highlight first receptive field
    rect = Rectangle((0.5, 0.5), 3, 3, fill=False, edgecolor='red', linewidth=3)
    ax1.add_patch(rect)
    ax1.text(2, -0.7, 'Receptive field\n(contributes to output[0,0])', 
             ha='center', fontsize=9, color='red', fontweight='bold')
    
    # Add colorbar
    plt.colorbar(im1, ax=ax1, fraction=0.046)
    ax1.set_xticks(range(5))
    ax1.set_yticks(range(5))
    ax1.grid(True, alpha=0.3, color='white', linewidth=0.5)
    
    # Kernel visualization
    ax2 = fig.add_subplot(gs[0, 1])
    im2 = ax2.imshow(kernel, cmap='RdBu', vmin=-0.5, vmax=0.5, interpolation='nearest')
    ax2.set_title('Convolutional Kernel (3×3)\n(Detects vertical edges/gradients)', 
                  fontweight='bold', fontsize=11)
    
    # Annotate kernel values
    for i in range(3):
        for j in range(3):
            text = ax2.text(j, i, f'{kernel[i, j]:.2f}',
                           ha='center', va='center', fontsize=10, fontweight='bold',
                           color='white' if abs(kernel[i, j]) > 0.2 else 'black')
    
    plt.colorbar(im2, ax=ax2, fraction=0.046)
    ax2.set_xticks(range(3))
    ax2.set_yticks(range(3))
    ax2.grid(True, alpha=0.3, color='gray', linewidth=0.5)
    
    # Output feature map
    ax3 = fig.add_subplot(gs[0, 2])
    im3 = ax3.imshow(output_data, cmap='plasma', interpolation='nearest')
    ax3.set_title('Output Feature Map (3×3)\n(Edge strength at each location)', 
                  fontweight='bold', fontsize=11)
    
    # Annotate output values
    for i in range(output_size):
        for j in range(output_size):
            text = ax3.text(j, i, f'{output_data[i, j]:.2f}',
                           ha='center', va='center', fontsize=10, fontweight='bold',
                           color='white' if abs(output_data[i, j]) < 0.5 else 'black')
    
    plt.colorbar(im3, ax=ax3, fraction=0.046)
    ax3.set_xticks(range(3))
    ax3.set_yticks(range(3))
    ax3.grid(True, alpha=0.3, color='white', linewidth=0.5)
    
    # ===== Middle Row: Computation breakdown for one output pixel =====
    ax4 = fig.add_subplot(gs[1, :])
    ax4.axis('off')
    ax4.set_xlim(0, 10)
    ax4.set_ylim(0, 2)
    
    # Extract the first 3x3 patch
    patch = input_data[0:3, 0:3]
    
    # Show computation
    computation_text = "Computing Output[0,0]:\n\n"
    computation_text += "Patch (input[0:3, 0:3])  ⊙  Kernel  =  Element-wise products  →  Sum\n\n"
    
    # Create visual equation
    patch_str = "["
    kernel_str = "["
    for i in range(3):
        if i > 0:
            patch_str += " "
            kernel_str += " "
        patch_str += " ".join([f"{patch[i,j]:5.2f}" for j in range(3)])
        kernel_str += " ".join([f"{kernel[i,j]:5.2f}" for j in range(3)])
        if i < 2:
            patch_str += "\n "
            kernel_str += "\n "
    patch_str += "]"
    kernel_str += "]"
    
    result = np.sum(patch * kernel)
    computation_text += f"{patch_str}  ⊙  {kernel_str}  =  {result:.4f}"
    
    ax4.text(5, 1, computation_text, ha='center', va='center', fontsize=9,
             fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # ===== Bottom Row: Dilated convolution comparison =====
    ax5 = fig.add_subplot(gs[2, 0])
    ax5.set_title('Standard 3×3 Convolution\nReceptive Field = 3×3', 
                  fontweight='bold', fontsize=11)
    
    # Create a larger input for dilation demo
    large_input = np.random.rand(7, 7)
    large_input[2:5, 2:5] += 0.5
    
    im5 = ax5.imshow(large_input, cmap='viridis', interpolation='nearest')
    # Highlight 3x3 receptive field
    rect_std = Rectangle((1.5, 1.5), 3, 3, fill=False, edgecolor='red', linewidth=3)
    ax5.add_patch(rect_std)
    ax5.set_xticks(range(7))
    ax5.set_yticks(range(7))
    ax5.grid(True, alpha=0.3, color='white', linewidth=0.5)
    
    # Mark kernel positions
    for i in range(2, 5):
        for j in range(2, 5):
            ax5.plot(j, i, 'r*', markersize=10)
    
    ax6 = fig.add_subplot(gs[2, 1])
    ax6.set_title('Dilated 3×3 (dilation=2)\nReceptive Field = 5×5', 
                  fontweight='bold', fontsize=11)
    
    im6 = ax6.imshow(large_input, cmap='viridis', interpolation='nearest')
    # Highlight 5x5 effective receptive field
    rect_dil = Rectangle((0.5, 0.5), 5, 5, fill=False, edgecolor='blue', linewidth=3)
    ax6.add_patch(rect_dil)
    ax6.set_xticks(range(7))
    ax6.set_yticks(range(7))
    ax6.grid(True, alpha=0.3, color='white', linewidth=0.5)
    
    # Mark sparse kernel positions (dilation = 2)
    positions = [(1, 1), (1, 3), (1, 5), 
                 (3, 1), (3, 3), (3, 5),
                 (5, 1), (5, 3), (5, 5)]
    for i, j in positions:
        ax6.plot(j, i, 'b*', markersize=10)
    
    ax7 = fig.add_subplot(gs[2, 2])
    ax7.axis('off')
    ax7.set_xlim(0, 1)
    ax7.set_ylim(0, 1)
    
    comparison_text = """
    Key Difference:
    
    Standard Convolution:
    • Kernel elements: contiguous
    • Receptive field: 3×3 = 9 pixels
    • Parameter count: 9 weights
    • Captures: local patterns only
    
    Dilated Convolution (dilation=2):
    • Kernel elements: spaced apart
    • Receptive field: 5×5 = 25 pixels
    • Parameter count: 9 weights (same!)
    • Captures: distant patterns
    
    For EM field prediction:
    → Standard: Detects slot edges
    → Dilated: Detects flux interactions
       across multiple slots
    """
    
    ax7.text(0.5, 0.5, comparison_text, ha='center', va='center', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    
    plt.suptitle('Convolutional Neural Network: Core Operations', 
                 fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n" + "="*80)
    print("CNN Architecture Summary for Magnetic Field Prediction")
    print("="*80)
    print()
    print("[Layer Type]       [Purpose]                        [EM Application]")
    print("-" * 80)
    print("Convolution        Feature extraction               Detect flux patterns")
    print("                   Local connectivity               Learn Maxwell's laws")
    print("                   Parameter sharing                Apply physics uniformly")
    print()
    print("Dilated Conv       Long-range interactions          Capture Biot-Savart law")
    print("                   Exponential receptive field      Model distant sources")
    print("                   Same parameter count             Efficient computation")
    print()
    print("Pooling            Spatial downsampling             Coarse field structure")
    print("                   Reduce parameters                Enable deep networks")
    print("                   Translation invariance           Robust to shifts")
    print()
    print("Skip Connections   Preserve spatial details         Sharp material boundaries")
    print("                   Multi-scale fusion               Airgap flux accuracy")
    print("                   Gradient flow                    Stable training")
    print()
    print("="*80)
    print("Result: 2.4M parameters predict 256×256 field in 10-100ms")
    print("        (vs. FEA: millions of mesh elements, 2-8 hours)")
    print("="*80)

visualize_convolution_detailed()

## Summary

This section established the CNN architecture for electromagnetic field prediction, detailing each component's role in capturing Maxwell's equations from data.

### Architectural Components

**1. Convolutional Layers** {cite}`lecun1998gradient`
- Local connectivity (3×3 filters) captures spatial relationships
- Parameter sharing applies physics laws uniformly across domain
- Feature hierarchy: geometric features → regional patterns → global topology

**2. Dilated Convolutions** {cite}`yu2015multi` *(Key Innovation)*
- Physics motivation: Biot-Savart law shows field depends on distant sources
- Receptive field grows exponentially (3×3 → 200×200) without parameter increase
- Enables long-range electromagnetic interactions

**3. Pooling Layers**
- Spatial downsampling (2×2, stride 2) reduces dimensions 75% per layer
- Enables deep networks to "see" entire motor geometry
- Trade-off: loses precision, recovered by skip connections

**4. Encoder-Decoder Structure** {cite}`ronneberger2015unet`
- Encoder: Progressive abstraction (256×256 → 32×32 bottleneck)
- Decoder: Progressive reconstruction (32×32 → 256×256 field prediction)
- Symmetric design balances compression with reconstruction

**5. Skip Connections** *(Critical for Accuracy)*
- Preserve spatial information lost during pooling
- Concatenate encoder features with decoder features
- Essential for sharp material boundaries (5× accuracy improvement)

### Network Specifications

| Component | Value |
|-----------|-------|
| Total layers | 32 |
| Trainable layers | 16 (convolutional) |
| Total parameters | 2.4 Million |
| Pooling/Upsampling | 8 (4 encoder + 4 decoder) |
| Dropout layers | 8 (regularization) |

### Key Insight for EM Field Prediction

The architecture exploits the structured 2D nature of magnetic fields through:
- **Spatial structure preservation**: CNNs maintain grid representation
- **Multi-scale physics**: Encoder-decoder mirrors local → regional → global scales
- **Long-range interactions**: Dilated convolutions capture non-local Biot-Savart effects
- **Precise localization**: Skip connections preserve material boundary positions

**Result**: 2.4M-parameter network predicts 256×256 fields in 10-100ms—crucial for design optimization requiring thousands of evaluations.

---

Section 4c examines the training process optimizing these 2.4 million parameters on 30,000 FEA-generated field distributions.